# 02 - Exploratory Analysis

## Objective
Explore overall delay behavior and identify early patterns to prioritize deeper SQL analysis.

## Load Clean Dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
df = pd.read_csv('../data/processed/vbz_delays_clean.csv')
df.head()

In [ ]:
df.shape

## Descriptive Statistics

In [ ]:
df[['departure_delay_sec', 'arrival_delay_sec', 'travel_time_diff_sec']].describe().T

## Departure Delay Distribution

In [ ]:
plt.figure(figsize=(10,4))
sns.histplot(df['departure_delay_sec'], bins=80, kde=True)
plt.xlim(-300, 1200)
plt.title('Distribution of Departure Delay (seconds)')
plt.xlabel('Departure delay (sec)')
plt.ylabel('Frequency')
plt.show()

## Delay by Hour of Day

In [ ]:
hourly = df.groupby('departure_hour', as_index=False)['departure_delay_sec'].mean()
hourly

In [ ]:
plt.figure(figsize=(10,4))
sns.lineplot(data=hourly, x='departure_hour', y='departure_delay_sec', marker='o', color='#d62728')
plt.title('Average Departure Delay by Hour')
plt.xlabel('Hour of day')
plt.ylabel('Average delay (sec)')
plt.xticks(range(0,24,1))
plt.show()

## Top Lines by Average Delay

In [ ]:
top_lines = (
    df.groupby('linie', as_index=False)['departure_delay_sec']
      .agg(['mean','count'])
      .reset_index()
      .rename(columns={'mean':'avg_delay_sec','count':'n'})
      .query('n >= 100')
      .sort_values('avg_delay_sec', ascending=False)
      .head(10)
)
top_lines

## Exploratory Insights
- Delay behavior is right-skewed: most segments are close to schedule, with a long tail of high-delay events.
- Delay rates are highest during low-volume night hours (01:00-02:00), while operationally relevant daytime pressure appears around 14:00-15:00.
- Delay burden is not uniform across lines; several lines consistently operate above the network average.
- These patterns justify targeted line/hour analysis in SQLite rather than global averages only.